# Day 5 - S3 from pandas

Read straight from the bucket, convert CSV to Parquet, write back, share with a presigned URL.

Everything here uses the `ds` CLI profile from Day 3. No keys in this file.

In [1]:
import time

import boto3
import pandas as pd

PROFILE = "ds"
BUCKET = "dave-ds-lab-ca"

# pandas -> s3fs -> boto. storage_options is how you tell s3fs which profile to use.
S3 = {"profile": PROFILE}

session = boto3.Session(profile_name=PROFILE)
s3 = session.client("s3")

def size_mb(key):
    """Object size in MB without downloading it (HEAD request)."""
    return s3.head_object(Bucket=BUCKET, Key=key)["ContentLength"] / 1e6

print("Identity:", session.client("sts").get_caller_identity()["Arn"])

Identity: arn:aws:iam::754928766714:user/D@ve_Admin


## 1. Read Parquet directly from S3

No download step. `s3://` is just another path to pandas once `s3fs` is installed.

In [2]:
PQ_RAW = f"s3://{BUCKET}/raw/yellow_tripdata_2024-01.parquet"

t0 = time.perf_counter()
df = pd.read_parquet(PQ_RAW, storage_options=S3)
t_parquet_read = time.perf_counter() - t0

print(f"{len(df):,} rows x {df.shape[1]} cols in {t_parquet_read:.1f}s")
df.head()

2,964,624 rows x 19 cols in 38.8s


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [3]:
df.dtypes

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
dtype: object

## 2. Write a CSV copy to `raw/`

We need a CSV in the bucket to compare against. Real life is usually the other way round - you
receive CSV and convert it - but the comparison is the same. This takes a few minutes: it is
~300 MB going up. (Rerun-safe: skips if it already exists.)

In [4]:
CSV_KEY = "raw/yellow_tripdata_2024-01.csv"
CSV_RAW = f"s3://{BUCKET}/{CSV_KEY}"

try:
    s3.head_object(Bucket=BUCKET, Key=CSV_KEY)
    print("CSV already in S3, skipping write")
except s3.exceptions.ClientError:
    t0 = time.perf_counter()
    df.to_csv(CSV_RAW, index=False, storage_options=S3)
    print(f"wrote CSV in {time.perf_counter() - t0:.0f}s")

print(f"CSV size: {size_mb(CSV_KEY):.0f} MB")

CSV already in S3, skipping write
CSV size: 314 MB


## 3. Read the CSV back, convert to Parquet (Snappy), write to `processed/`

Watch two things: how long the CSV read takes vs the Parquet read above, and what happens to the
dtypes (CSV has no schema - timestamps come back as strings).

In [5]:
t0 = time.perf_counter()
df_csv = pd.read_csv(CSV_RAW, storage_options=S3)
t_csv_read = time.perf_counter() - t0
print(f"CSV read: {t_csv_read:.1f}s  (Parquet read was {t_parquet_read:.1f}s)")

# CSV lost the timestamp types - put them back before writing Parquet.
for col in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]:
    df_csv[col] = pd.to_datetime(df_csv[col])

df_csv.dtypes

C:\Users\Diomi\AppData\Local\Temp\ipykernel_66372\3180026029.py:2: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df_csv = pd.read_csv(CSV_RAW, storage_options=S3)


CSV read: 348.1s  (Parquet read was 38.8s)


VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
dtype: object

In [6]:
PQ_KEY = "processed/yellow_tripdata_2024-01.parquet"
PQ_PROCESSED = f"s3://{BUCKET}/{PQ_KEY}"

t0 = time.perf_counter()
df_csv.to_parquet(PQ_PROCESSED, compression="snappy", index=False, storage_options=S3)
print(f"wrote Parquet in {time.perf_counter() - t0:.0f}s")

wrote Parquet in 46s


## 4. The size comparison

This is the number to remember. Same rows, same columns.

In [7]:
csv_mb = size_mb(CSV_KEY)
pq_mb = size_mb(PQ_KEY)

print(f"CSV      raw/        {csv_mb:7.1f} MB   read {t_csv_read:5.1f}s")
print(f"Parquet  processed/  {pq_mb:7.1f} MB   read {t_parquet_read:5.1f}s")
print(f"\nParquet is {csv_mb / pq_mb:.1f}x smaller.")
print("On Athena (Day 9) that ratio is also roughly the cost ratio - it bills per byte scanned.")

CSV      raw/          313.6 MB   read 348.1s
Parquet  processed/     60.8 MB   read  38.8s

Parquet is 5.2x smaller.
On Athena (Day 9) that ratio is also roughly the cost ratio - it bills per byte scanned.


## 5. Presigned URL - share a private object without making it public

The URL carries a signature made with *your* credentials and expires. Anyone holding it can
download that one object until then. The bucket stays private.

In [8]:
url = s3.generate_presigned_url(
    "get_object",
    Params={"Bucket": BUCKET, "Key": PQ_KEY},
    ExpiresIn=600,  # seconds
)
print(url)
print("\nPaste this in an incognito window: it downloads. Wait 10 minutes and try again: AccessDenied.")

https://dave-ds-lab-ca.s3.amazonaws.com/processed/yellow_tripdata_2024-01.parquet?X-Amz-...  (signature trimmed before commit)

Paste this in an incognito window: it downloads. Wait 10 minutes and try again: AccessDenied.


## 6. What is in the bucket now

In [9]:
resp = s3.list_objects_v2(Bucket=BUCKET)
for obj in resp.get("Contents", []):
    if obj["Size"]:
        print(f"{obj['Size'] / 1e6:8.1f} MB  {obj['Key']}")

    60.8 MB  processed/yellow_tripdata_2024-01.parquet
   313.6 MB  raw/yellow_tripdata_2024-01.csv
    50.0 MB  raw/yellow_tripdata_2024-01.parquet
    50.3 MB  raw/yellow_tripdata_2024-02.parquet
    60.1 MB  raw/yellow_tripdata_2024-03.parquet
